In [10]:
# Clean environment and install
!pip uninstall googletrans -y -q
!pip install gradio deep-translator gTTS --quiet --root-user-action=ignore
import warnings
warnings.filterwarnings('ignore')
print(" Environment cleared and all dependencies installed ")

 Environment cleared and all dependencies installed 


In [16]:
# Corrected Gradio Translation Application Code
import gradio as gr
from deep_translator import GoogleTranslator
from gtts import gTTS
import os

# 1. Dynamically fetch the supported languages dictionary from the backend
translator_instance = GoogleTranslator()
supported_languages = translator_instance.get_supported_languages(as_dict=True)

# Format the list nicely for our UI dropdown selection (e.g., "english", "urdu")
ui_languages_list = sorted(list(supported_languages.keys()))

# Backend Logic: API Communication & Feature Processing
def process_translation_task(input_text, source_language, target_language):
    # Validation check for empty input fields
    if not input_text.strip():
        return " Validation Error: Please enter some text to translate.", None

    try:
        # Fetching standard ISO codes from our dynamic dictionary
        src_iso = supported_languages[source_language.lower()]
        tgt_iso = supported_languages[target_language.lower()]

        # 2 & 3. Connect to the Translation API and get the response
        translated_result = GoogleTranslator(source=src_iso, target=tgt_iso).translate(input_text)

        # 5. Optional Feature: Text-to-Speech (TTS) Engine Generation
        audio_file_path = "translated_speech.mp3"
        try:
            # Generate localized voice responses using the target ISO code
            tts_engine = gTTS(text=translated_result, lang=tgt_iso, slow=False)
            tts_engine.save(audio_file_path)
            audio_output = audio_file_path
        except Exception:
            # Safe fallback if the specific language voice font is unavailable on the server
            audio_output = None

        return translated_result, audio_output

    except Exception as api_error:
        return f" Translation API Error: {str(api_error)}", None

# 4. Building the User Interface Layout (Requirement 1)
with gr.Blocks(theme=gr.themes.Default(primary_hue="blue", secondary_hue="gray")) as internship_app:

    gr.Markdown("# 🌐 AI Multi-Language Translator Pro")
    gr.Markdown("---")

    with gr.Row():
        source_dropdown = gr.Dropdown(choices=ui_languages_list, value="english", label="Select Source Language (From)")
        target_dropdown = gr.Dropdown(choices=ui_languages_list, value="urdu", label="Select Target Language (To)")

    with gr.Row():
        text_input = gr.Textbox(placeholder="Type or paste your text here...", label="Input Text", lines=5)

    # Main action submission element
    submit_button = gr.Button("Process & Translate Text", variant="primary")

    gr.Markdown("---")
    gr.Markdown("### 📝 Results & Accessibility Controls")

    with gr.Row():
        # Clean Output Screen Display (Requirement 4) with integrated 1-click Copy Module (Requirement 5)
        text_output = gr.Textbox(label="Translated Text (Click top-right icon to Copy 📋)", lines=5, show_copy_button=True)
        # Audio Player Component for accessibility listening (Requirement 5)
        audio_output = gr.Audio(label="🔊 Text-to-Speech Player", type="filepath")

    # Mapping interactive trigger bindings
    submit_button.click(
        fn=process_translation_task,
        inputs=[text_input, source_dropdown, target_dropdown],
        outputs=[text_output, audio_output]
    )

# Launching application with auto-sharing links generated dynamically
internship_app.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9dcf4f325674674f66.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
